# 1. Load Data

In [1]:
import pandas as pd
import numpy as np

def load_data(data_path : str):
    df = pd.read_csv(f"{data_path}")

    return df

def save_data(df: pd.DataFrame, data_path : str):

    return df.to_csv(f"{data_path}", index = False)

previous_application = pd.read_csv("D:/4. Home Credit/Dataset/CSV/previous_application.csv")

In [2]:
previous_application_copy = previous_application.copy()
previous_application_copy.head()

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


# 2. Customized One Hot Encoder Function

In [3]:
def OHE(df : pd.DataFrame):
    # Get categorical columns
    categorical_columns = list(df.select_dtypes(include = ['object']))

    # OHE 
    df = pd.get_dummies(df, columns = categorical_columns, dummy_na = False, drop_first = False )

    # Return df
    return df 

# 3. Preprocessing, Feature Engineering and Aggregation

### Preprocessing

In [4]:
ohe_list = list(previous_application_copy.select_dtypes(include = ['object']))
previous_application_copy = previous_application_copy.drop(columns = ['SK_ID_PREV', "NAME_SELLER_INDUSTRY", "PRODUCT_COMBINATION"])
previous_application_copy = OHE(previous_application_copy)

C:\Users\Admin\AppData\Local\Temp\ipykernel_24184\2339028248.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  ohe_list = list(previous_application_copy.select_dtypes(include = ['object']))
C:\Users\Admin\AppData\Local\Temp\ipykernel_24184\312675149.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pyda

In [5]:
previous_application_copy.head()

,SK_ID_CURR,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,HOUR_APPR_PROCESS_START,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,...,CHANNEL_TYPE_Contact center,CHANNEL_TYPE_Country-wide,CHANNEL_TYPE_Credit and cash offices,CHANNEL_TYPE_Regional / Local,CHANNEL_TYPE_Stone,NAME_YIELD_GROUP_XNA,NAME_YIELD_GROUP_high,NAME_YIELD_GROUP_low_action,NAME_YIELD_GROUP_low_normal,NAME_YIELD_GROUP_middle
0,271877,1730.430,17145.0,17145.0,0.0,17145.0,15,1,0.0,0.182832,...,False,True,False,False,False,False,False,False,False,True
1,108129,25188.615,607500.0,679671.0,NaN,607500.0,11,1,NaN,NaN,...,True,False,False,False,False,False,False,True,False,False
2,122040,15060.735,112500.0,136444.5,NaN,112500.0,11,1,NaN,NaN,...,False,False,True,False,False,False,True,False,False,False
3,176158,47041.335,450000.0,470790.0,NaN,450000.0,7,1,NaN,NaN,...,False,False,True,False,False,False,False,False,False,True
4,202054,31924.395,337500.0,404055.0,NaN,337500.0,9,1,NaN,NaN,...,False,False,True,False,False,False,True,False,False,False


### Feature Engineering

In [6]:
# Days
cols = ['DAYS_FIRST_DRAWING','DAYS_FIRST_DUE','DAYS_LAST_DUE_1ST_VERSION','DAYS_LAST_DUE','DAYS_TERMINATION']
previous_application_copy[cols] = previous_application_copy[cols].replace(365243, np.nan)

# Annuity
previous_application_copy['ANNUITY_TO_CREDIT_RATIO'] = previous_application_copy['AMT_ANNUITY'] / (previous_application_copy['AMT_CREDIT'] + 1e-9)
previous_application_copy['ANNUITY_TO_APPLICATION_RATIO'] = previous_application_copy['AMT_ANNUITY'] / (previous_application_copy['AMT_APPLICATION'] + 1e-9)

# Months need to pay off the debt
previous_application_copy['MONTHS_TO_PAY_OFF'] = previous_application_copy['AMT_GOODS_PRICE'] / (previous_application_copy['AMT_ANNUITY'] + 1e-9)

# Credit
previous_application_copy['CREDIT_TO_GOODS_RATIO'] = previous_application_copy['AMT_CREDIT'] / (previous_application_copy['AMT_GOODS_PRICE'] + 1e-9)
previous_application_copy['AMT_PAYMENT_PER_TIME'] = previous_application_copy['AMT_CREDIT'] / (previous_application_copy['CNT_PAYMENT'] + 1e-9)

# Downpayment
previous_application_copy['DOWNPAYMENT_RATIO'] = previous_application_copy['AMT_DOWN_PAYMENT'] / (previous_application_copy['AMT_CREDIT'] + 1e-9)

# DLD
previous_application_copy['DAYS_LAST_DUE_DIFF'] = previous_application_copy['DAYS_LAST_DUE_1ST_VERSION'] - previous_application_copy['DAYS_LAST_DUE']
previous_application_copy['PLANNED_DURATION'] = previous_application_copy['DAYS_LAST_DUE_1ST_VERSION'] - previous_application_copy['DAYS_FIRST_DUE']
previous_application_copy['ACTUAL_DURATION'] = previous_application_copy['DAYS_LAST_DUE'] - previous_application_copy['DAYS_FIRST_DUE']
previous_application_copy['PREVIOUS_APPLICATION_IS_ACTIVE'] = previous_application_copy['DAYS_LAST_DUE'].isna() & previous_application_copy['DAYS_FIRST_DUE'].notna()
previous_application_copy['TERMINATION_MINUS_LAST_DUE'] = previous_application_copy['DAYS_TERMINATION'] - previous_application_copy['DAYS_LAST_DUE']
previous_application['EARLY_TERMINATION'] = previous_application_copy['DAYS_LAST_DUE_1ST_VERSION'] - previous_application_copy['DAYS_TERMINATION']


# L1M, L3M, L6M
recent_features = []
for month in [1,3,6]:
    recent = previous_application_copy[previous_application_copy['DAYS_DECISION'].between(-30 * month, 0)]
    agg = recent.groupby("SK_ID_CURR").agg(**{
        f"PREVIOUS_APPLICATION_AMT_APPLICATION_L{month}M_MEAN" : ('AMT_APPLICATION', 'mean'),
        f"PREVIOUS_APPLICATION_AMT_APPLICATION_L{month}M_SUM" : ('AMT_APPLICATION', 'sum'),
        f"PREVIOUS_APPLICATION_AMT_APPLICATION_L{month}M_MAX" : ('AMT_APPLICATION', 'max'),
        f"PREVIOUS_APPLICATION_APPROVED_AMT_CREDIT_L{month}M_MEAN" : ('AMT_CREDIT', 'mean'),
        f"PREVIOUS_APPLICATION_APPROVED_AMT_CREDIT_L{month}M_SUM" : ('AMT_CREDIT', 'sum'),
        f"PREVIOUS_APPLICATION_APPROVED_AMT_CREDIT_L{month}M_MAX" : ('AMT_CREDIT', 'max'),
        f"PREVIOUS_APPLICATION_APPROVED_AMT_ANNUITY_L{month}M_MEAN" : ('AMT_ANNUITY', 'mean'),
        f"PREVIOUS_APPLICATION_AAPPROVED_AMT_ANNUITY_L{month}M_MAX" : ('AMT_ANNUITY', 'max'),
    })
    recent_features.append(agg)

recent_features = pd.concat(recent_features, axis = 1).reset_index()


# CASH_LOAN
cash_loan = []
previous_application_copy['IS_CASH_LOAN'] = previous_application_copy['NAME_CONTRACT_TYPE_Cash loans'].astype(int)
for flag, name in [('IS_CASH_LOAN', "CASH_LOAN")]:
    sub = previous_application_copy[previous_application_copy[flag] == 1]
    agg = sub.groupby("SK_ID_CURR").agg(**{
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_SUM" : ('AMT_CREDIT', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_SUM" : ('AMT_ANNUITY', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_APPLICATION_SUM" : ('AMT_APPLICATION', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_GOODS_PRICE_SUM" : ('AMT_GOODS_PRICE', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MAX" : ('AMT_CREDIT', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MAX" : ('AMT_ANNUITY', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MEAN" : ('AMT_CREDIT', 'mean'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MEAN" : ('AMT_CREDIT', 'mean')
    })
    cash_loan.append(agg)

cash_loan = pd.concat(cash_loan, axis = 1).reset_index()

# CONSUMER LOANS
consumer_loan = []
previous_application_copy['IS_CONSUMER_LOAN'] = previous_application_copy['NAME_CONTRACT_TYPE_Consumer loans'].astype(int)
for flag, name in [('IS_CONSUMER_LOAN', "CONSUMER_LOAN")]:
    sub = previous_application_copy[previous_application_copy[flag] == 1]
    agg = sub.groupby("SK_ID_CURR").agg(**{
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_SUM" : ('AMT_CREDIT', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_SUM" : ('AMT_ANNUITY', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_APPLICATION_SUM" : ('AMT_APPLICATION', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_GOODS_PRICE_SUM" : ('AMT_GOODS_PRICE', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MAX" : ('AMT_CREDIT', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MAX" : ('AMT_ANNUITY', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MEAN" : ('AMT_CREDIT', 'mean'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MEAN" : ('AMT_CREDIT', 'mean')
    })
    consumer_loan.append(agg)

consumer_loan = pd.concat(consumer_loan, axis = 1).reset_index()


# APPROVED PREVIOUS APPLICATION
approved = []
previous_application_copy['IS_APPROVED'] = previous_application_copy['NAME_CONTRACT_STATUS_Approved'].astype(int)
for flag, name in [('IS_APPROVED', "APPROVED")]:
    sub = previous_application_copy[previous_application_copy[flag] == 1]
    agg = sub.groupby("SK_ID_CURR").agg(**{
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_SUM" : ('AMT_CREDIT', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_SUM" : ('AMT_ANNUITY', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_APPLICATION_SUM" : ('AMT_APPLICATION', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_GOODS_PRICE_SUM" : ('AMT_GOODS_PRICE', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MAX" : ('AMT_CREDIT', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MAX" : ('AMT_ANNUITY', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MEAN" : ('AMT_CREDIT', 'mean'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MEAN" : ('AMT_CREDIT', 'mean')
    })
    approved.append(agg)

approved = pd.concat(approved, axis = 1).reset_index()

# CANCELED PREVIOUS APPLICATION
canceled = []
previous_application_copy['IS_CANCELED'] = previous_application_copy['NAME_CONTRACT_STATUS_Canceled'].astype(int)
for flag, name in [('IS_CANCELED', "CANCELED")]:
    sub = previous_application_copy[previous_application_copy[flag] == 1]
    agg = sub.groupby("SK_ID_CURR").agg(**{
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_SUM" : ('AMT_CREDIT', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_SUM" : ('AMT_ANNUITY', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_APPLICATION_SUM" : ('AMT_APPLICATION', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_GOODS_PRICE_SUM" : ('AMT_GOODS_PRICE', 'sum'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MAX" : ('AMT_CREDIT', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MAX" : ('AMT_ANNUITY', 'max'),
        f"PREVIOUS_APPLICATION_{name}_AMT_CREDIT_MEAN" : ('AMT_CREDIT', 'mean'),
        f"PREVIOUS_APPLICATION_{name}_AMT_ANNUITY_MEAN" : ('AMT_CREDIT', 'mean')
    })
    canceled.append(agg)

canceled = pd.concat(canceled, axis = 1).reset_index()

### Aggregation

In [7]:
# Define aggregation dictionaries
previous_application_numeric_aggregation = {}
previous_application_ohe_aggregation = {}


# Numeric Aggregation
previous_application_numeric_aggregation = {
    'AMT_ANNUITY' : ['mean','max','min'],
    'AMT_APPLICATION': ['mean','max','min'],
    'AMT_CREDIT': ['mean','max','min'],
    'AMT_DOWN_PAYMENT': ['mean','max','sum'],
    'AMT_GOODS_PRICE': ['mean','max','sum'],
    'RATE_DOWN_PAYMENT': ['min', 'max', 'mean'],
    'DAYS_DECISION': ['min', 'max', 'mean'],
    'CNT_PAYMENT': ['mean', 'sum','min','max'],
    'ANNUITY_TO_CREDIT_RATIO': ['mean','max','min'],
    'ANNUITY_TO_APPLICATION_RATIO' : ['mean','max','min'],
    'MONTHS_TO_PAY_OFF': ['mean','max','min'],
    'CREDIT_TO_GOODS_RATIO': ['mean','max','min'],
    'AMT_PAYMENT_PER_TIME' : ['mean','max','min'],
    'DOWNPAYMENT_RATIO': ['mean','max','min'],
    'DAYS_TERMINATION' : ['mean', 'max'],
    'DAYS_LAST_DUE' : ['mean', 'max'],
    'DAYS_LAST_DUE_1ST_VERSION' : ['mean', 'max'],
    'DAYS_FIRST_DUE' : ['mean', 'max'],
    'DAYS_LAST_DUE_DIFF' : ['mean', 'max'],
    'PLANNED_DURATION' : ['mean','max']
}


# OHE Aggregation
ohe_cols = [col for col in previous_application_copy.columns if col.startswith(tuple(ohe_list))]
previous_application_ohe_aggregation = {
    col : ['mean'] for col in ohe_cols
}


# Aggregation 
previous_application_agg = previous_application_copy.groupby("SK_ID_CURR").agg({**previous_application_numeric_aggregation, **previous_application_ohe_aggregation})
previous_application_agg.columns = [f"PREVIOUS_APPLICATION_{col}_{status}".upper() for col, status in previous_application_agg.columns]
previous_application_agg = previous_application_agg.reset_index()
previous_application_agg = previous_application_agg.merge(recent_features, on = 'SK_ID_CURR', how = 'left')
previous_application_agg = previous_application_agg.merge(cash_loan, on = 'SK_ID_CURR', how = 'left')
previous_application_agg = previous_application_agg.merge(consumer_loan, on = 'SK_ID_CURR', how = 'left')
previous_application_agg = previous_application_agg.merge(approved, on = 'SK_ID_CURR', how = 'left')
previous_application_agg = previous_application_agg.merge(canceled, on = 'SK_ID_CURR', how = 'left')
previous_application_agg.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_24184\3407964366.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  previous_application_agg = previous_application_agg.reset_index()


,SK_ID_CURR,PREVIOUS_APPLICATION_AMT_ANNUITY_MEAN,PREVIOUS_APPLICATION_AMT_ANNUITY_MAX,PREVIOUS_APPLICATION_AMT_ANNUITY_MIN,PREVIOUS_APPLICATION_AMT_APPLICATION_MEAN,PREVIOUS_APPLICATION_AMT_APPLICATION_MAX,PREVIOUS_APPLICATION_AMT_APPLICATION_MIN,PREVIOUS_APPLICATION_AMT_CREDIT_MEAN,PREVIOUS_APPLICATION_AMT_CREDIT_MAX,PREVIOUS_APPLICATION_AMT_CREDIT_MIN,...,PREVIOUS_APPLICATION_APPROVED_AMT_CREDIT_MEAN,PREVIOUS_APPLICATION_APPROVED_AMT_ANNUITY_MEAN,PREVIOUS_APPLICATION_CANCELED_AMT_CREDIT_SUM,PREVIOUS_APPLICATION_CANCELED_AMT_ANNUITY_SUM,PREVIOUS_APPLICATION_CANCELED_AMT_APPLICATION_SUM,PREVIOUS_APPLICATION_CANCELED_AMT_GOODS_PRICE_SUM,PREVIOUS_APPLICATION_CANCELED_AMT_CREDIT_MAX,PREVIOUS_APPLICATION_CANCELED_AMT_ANNUITY_MAX,PREVIOUS_APPLICATION_CANCELED_AMT_CREDIT_MEAN,PREVIOUS_APPLICATION_CANCELED_AMT_ANNUITY_MEAN
0,100001,3951.000,3951.000,3951.000,24835.50,24835.5,24835.5,23787.00,23787.0,23787.0,...,23787.0,23787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100002,9251.775,9251.775,9251.775,179055.00,179055.0,179055.0,179055.00,179055.0,179055.0,...,179055.0,179055.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100003,56553.990,98356.995,6737.310,435436.50,900000.0,68809.5,484191.00,1035882.0,68053.5,...,484191.0,484191.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100004,5357.250,5357.250,5357.250,24282.00,24282.0,24282.0,20106.00,20106.0,20106.0,...,20106.0,20106.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,4813.200,4813.200,4813.200,22308.75,44617.5,0.0,20076.75,40153.5,0.0,...,40153.5,40153.5,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0


# 4. Save Data

In [8]:
save_data(previous_application_agg, "D:/4. Home Credit/Dataset/CSV/previous_application_final_agg.csv")